# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [10]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [19]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
)

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [1]:
links = fetch_website_links("https://edwarddonner.com")
links

NameError: name 'fetch_website_links' is not defined

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [21]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [22]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [23]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [24]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [25]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills / proficient page',
   'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog / posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'brand page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [26]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'company page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [27]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [28]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-35B-A3B
Updated
about 6 hours ago
•
259k
•
613
Qwen/Qwen3.5-27B
Updated
3 days ago
•
108k
•
398
Qwen/Qwen3.5-397B-A17B
Updated
4 days ago
•
726k
•
1.11k
Qwen/Qwen3.5-122B-A10B
Updated
3 days ago
•
108k
•
325
unsloth/Qwen3.5-35B-A3B-GGUF
Updated
3 days ago
•
265k
•
276
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.72k
Qwen Image Multiple Angles 3D Camera
🎥
1.72k
Change the camera angle of a photo with AI
Running
on
Zero
MCP
979
Wan2.2 14B Preview
🐌
979
generate a video from an image with a text prompt
Running
on
Zero
Featured
192
Omni Video Factory
🏆
192
text to video, image to video, video extend
Running
on
Zer

In [29]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [30]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [31]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\nabout 6 hours ago\n•\n259k\n•\n613\nQwen/Qwen3.5-27B\nUpdated\n3 days ago\n•\n108k\n•\n399\nQwen/Qwen3.5-397B-A17B\nUpdated\n4 days ago\n•\n726k\n•\n1.11k\nQwen/Qwen3.5-122B-A10B\nUpdated\n3 days ago\n•\n108k\n•\n325\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\n3 days ago\n•\n265k\n•\n276\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.72k\nQwen Image Mu

In [32]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [33]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a vibrant AI community and platform dedicated to building the future of machine learning. It serves as a collaborative hub where machine learning enthusiasts, researchers, and developers come together to create, discover, share, and improve models, datasets, and applications.

By hosting over **2 million models** and **500k+ datasets**, Hugging Face has established itself as the go-to platform for AI developers to accelerate their ML projects across various modalities, including text, image, video, audio, and even 3D.

---

## What Hugging Face Offers

- **Models:** Access and contribute to a vast library of machine learning models, including state-of-the-art large language models like Qwen and demo applications for image and video generation.
- **Datasets:** Use and share diverse datasets for training and benchmarking models, with active updates and community engagement.
- **Spaces:** Deploy and showcase interactive AI applications and demos running on the platform’s infrastructure.
- **Community:** Connect with a global network of AI practitioners, collaborate on projects, and build your ML portfolio.
- **Docs & Tutorials:** Comprehensive documentation and resources ensuring users can get started quickly and develop efficiently.
- **Enterprise Solutions:** Tailored offerings for businesses looking to leverage AI at scale with support and integration services.
- **Open Source Stack:** Accelerate development with access to free, open-source machine learning tools and libraries.

---

## Company Culture

Hugging Face fosters an open, collaborative culture driven by community and innovation. The platform encourages sharing and transparency, empowering creators to build and showcase their work while learning from peers. The company prides itself on inclusivity and fostering a space where cutting-edge AI solutions can be developed democratically.

---

## Customers and Users

- Research institutions and universities leveraging the platform for academic ML research.
- AI developers and startups building novel applications in NLP, computer vision, speech, and other AI fields.
- Large enterprises using Hugging Face’s enterprise offering for AI model deployment and maintenance.
- Data scientists and ML engineers growing their portfolios and utilizing community-shared assets for faster iteration.

---

## Careers at Hugging Face

Hugging Face consistently seeks passionate AI and ML professionals to join their expanding team. Opportunities span engineering, research, product development, and community roles. Joining means contributing directly to the future of AI innovation within an open and community-focused environment.

For current openings and job descriptions, visit their careers page at [Hugging Face Careers](https://huggingface.co/careers).

---

## Join the Community!

Whether you're a seasoned data scientist, developer, or newcomer eager to dive into AI, Hugging Face provides the tools, resources, and collaborative spirit to help you thrive.

- **Sign up** to explore over 2M+ models, 500k+ datasets, and 1M+ AI applications.
- Share your projects and build your ML profile.
- Join discussions and collaborate with peers worldwide.

Explore more at [https://huggingface.co](https://huggingface.co) and be part of the AI community building the future!

---

Hugging Face — Empowering a collaborative, open AI future.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [37]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [38]:
stream_brochure("HuggingFace", "https://huggingface.co")

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future. It serves as the premier collaboration platform for the global machine learning (ML) community, enabling users to explore, share, and innovate on ML models, datasets, and applications. The platform hosts over 2 million models and 500,000+ datasets spanning multiple modalities including text, image, video, audio, and even 3D.

At its core, Hugging Face empowers machine learning engineers, scientists, and end users to build an open and ethical AI future through community-driven innovation. The company's open-source libraries and tools are widely used and continuously enhanced by a rapidly growing and vibrant AI community.

---

## Platform and Offerings

- **Hugging Face Hub:** A central place for hosting, discovering, and collaborating on public and private machine learning assets including models, datasets, and applications.
- **Spaces:** Interactive demos and applications running on the platform showcasing cutting-edge AI technology.
- **Models:** Access to over 2 million ML models including recent popular ones like Qwen-series language models.
- **Datasets:** Extensive datasets enabling a wide range of AI research and application development.
- **Open Source Stack:** Tools and libraries designed to accelerate machine learning development and deployment.
- **Multi-modal Support:** Support for various data types such as textual, audio, imagery, video, and 3D.

---

## Enterprise Solutions

Hugging Face extends its platform for organizations with the Enterprise Hub offering advanced capabilities:

- **Enterprise-grade Security:** Single Sign-On integration, audit logging, and granular access control.
- **Data Management:** Regional repository management, private storage with additional 1 TB per user, and private dataset viewers.
- **Scalability:** Advanced compute options with ZeroGPU quota boosts and multiple inference providers.
- **Analytics:** Comprehensive dashboards to track repository and usage data.
- **Support:** Dedicated enterprise-level customer support and flexible contract options.

Pricing starts at $20 per user/month for team subscriptions, with tailored enterprise plans available.

---

## Company Culture

Hugging Face fosters a culture of openness, collaboration, and innovation. The community-centric approach champions ethical AI development and knowledge sharing. Its users are encouraged to build portfolios, share their work, and contribute to the open source ecosystem. Hugging Face supports the next generation of AI talent, blending technical excellence with community engagement.

---

## Careers

Hugging Face is a growing company looking for talented individuals passionate about AI and open source. The team is composed of scientists, engineers, and community builders dedicated to pushing the boundaries of machine learning technology. Career opportunities span research, software engineering, community management, and enterprise support roles. Joining Hugging Face means contributing to a global AI revolution in an ethical and collaborative environment.

---

## Who Uses Hugging Face?

- **Machine Learning Researchers** looking for easy access and sharing of cutting-edge models and datasets.
- **AI Engineers** building AI-powered applications across different modalities.
- **Data Scientists and Analysts** leveraging vast, diverse datasets for experimentation.
- **Enterprises** needing scalable AI platforms with security and manageability features.
- **Educators and Students** using the platform to learn and teach machine learning.

---

## Connect and Learn More

- Explore AI Apps & Models: [huggingface.co](https://huggingface.co)
- Join the Community on GitHub, Twitter, Discord, and LinkedIn
- Learn & Collaborate through Docs and Forum on the website
- Contact Sales for Enterprise Solutions

---

### Brand Palette

- Hugging Face Yellow: #FFD21E  
- Accent Orange: #FF9D00  
- Neutral Gray: #6B7280

---

**Hugging Face** — The AI community building the future. Be part of the journey to create open, collaborative, and responsible AI.  

Sign up today and accelerate your machine learning innovation!

In [39]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face Brochure

## About Hugging Face
Hugging Face is a leading AI company and collaboration platform dedicated to building the future of machine learning. It serves as the central hub where the global machine learning community collaborates, sharing and exploring open-source models, datasets, and AI applications across diverse modalities including text, image, video, audio, and 3D.

## What We Offer
- **Extensive Model Repository**: Access over 2 million machine learning models, including the latest trending models updated continuously by the community.
- **Rich Dataset Library**: Discover and contribute to over 500,000 datasets catering to a wide range of ML tasks.
- **Spaces**: Deploy, demo, and experiment with interactive AI applications hosted on the platform.
- **Open Source Stack**: Leverage Hugging Face's open source tools to accelerate machine learning development and deployment.
- **Collaboration Tools**: Host unlimited public models, datasets, and applications to build your portfolio and collaborate with a worldwide community.

## Our Community and Customers
Hugging Face empowers a diverse ecosystem consisting of machine learning engineers, data scientists, researchers, and organizations. The platform supports:
- Independent AI developers building portfolios and sharing their work
- Research communities creating cutting-edge machine learning libraries
- Enterprises seeking to build and deploy AI applications with ethical and open-source solutions
- Educators and learners promoting accessible AI education and knowledge sharing

## Company Culture
Hugging Face fosters an open and ethical AI future built on transparency, collaboration, and inclusiveness. The company encourages sharing knowledge freely to empower the next generation of AI innovation. The fast-growing community thrives on trust, mutual support, and innovation.

## Careers at Hugging Face
Join Hugging Face to be part of the AI revolution. The company regularly offers positions for:
- Machine Learning Engineers
- Research Scientists
- Software Developers
- Community Managers and Content Creators
- Enterprise AI Solutions Architects  
  
Work here means engaging with a vibrant community, working on cutting-edge AI projects, and contributing to an open-source ecosystem that impacts millions globally.

---

## Get Involved
- Explore AI apps and browse models at the [Hugging Face Hub](https://huggingface.co)
- Share your own projects and datasets to build your professional ML portfolio
- Join the vibrant global AI community to learn, collaborate, and innovate together

**Hugging Face: The AI community building the future.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>